# 1. A consumer with two nests

### Approach for Section 1.1

The consumer's problem is originally stated in quantities $(x_1, x_2, x_3)$
subject to the linear budget constraint

$$
p_1 x_1 + p_2 x_2 + p_3 x_3 = I,
$$

which a box-constrained solver like L-BFGS-B can't handle directly. Section
1.1 reparametrizes the choice into two numbers confined to $[0,1]$: $s_1$,
the share of income spent on food, and $w$, the share of the remaining
travel budget spent on the bus. The three budget shares then follow
automatically as

$$
s_1, \qquad s_2 = (1 - s_1) w, \qquad s_3 = (1 - s_1)(1 - w),
$$

which always sum to one, so the budget constraint holds for any choice in
the unit square — turning the constrained problem into a simple box
$[0,1] \times [0,1]$, exactly what L-BFGS-B takes as bounds, with no
penalty terms needed.

This mapping (`shares` and `quantities`) is already implemented in
`Consumer.py`, so our job is to understand why it works (linking to the
reparametrization idea from section 6.4 of the optimization lecture) and to
build the utility side of the problem on top of it:

- `utility(x1, x2, x3)` implements the nested CES formula

$$
x_B = \left[\beta x_2^{\rho_B} + (1-\beta) x_3^{\rho_B}\right]^{\frac{1}{\rho_B}},
\qquad
u(x_1, x_2, x_3) = \left[\alpha x_1^{\rho_A} + (1-\alpha) x_B^{\rho_A}\right]^{\frac{1}{\rho_A}},
$$

  combining goods 2 and 3 into the travel composite $x_B$ first, and then
  combining that with good 1.

- `value_of_choice(s1, w)` turns the nested shares into quantities using
  the given `quantities()` function and passes them to `utility()`, giving
  the objective that `solve_grid` and `solve` will later maximize.

In short, 1.1 sets up *what* is being maximized as a function of $(s_1, w)$,
not yet *how* it gets solved.

In [3]:
def utility(self,x1,x2,x3):
    """ nested CES utility of a bundle of quantities

    Two steps: first combine goods 2 and 3 into the travel composite, then
    combine good 1 and the composite into utility. Use .ces() for both.

    Args:

        x1 (float or ndarray): quantity of good 1
        x2 (float or ndarray): quantity of good 2
        x3 (float or ndarray): quantity of good 3

    Returns:

        (float or ndarray): utility

    """

    par = self.par

    # a. travel composite (nest of bus and train)
    xB = self.ces(x2,x3,par.beta,par.sigma_B)

    # b. utility (nest of food and travel)
    u = self.ces(x1,xB,par.alpha,par.sigma_A)

    return u

In [4]:
def value_of_choice(self,s1,w):
    """ utility of the bundle implied by the nested shares

    Args:

        s1 (float or ndarray): share of income spent on food
        w (float or ndarray): share of the travel budget spent on the bus

    Returns:

        (float or ndarray): utility

    """

    # a. quantities implied by the nested shares
    x1,x2,x3 = self.quantities(s1,w)

    # b. utility of that bundle
    u = self.utility(x1,x2,x3)

    return u

In 1.1 we must rewrite the consumers optimization problem, so that Python choose two numbers (s1, w) instead of three quantities (x1, x2, x3)

Test Question 1

In [6]:
# Whoopi sanity check

model = ConsumerClass()
for s1_now,w_now in [(0.0,0.0),(0.3,0.5),(0.6,0.7),(1.0,1.0)]:
    x1,x2,x3 = model.quantities(s1_now,w_now)
    expenditure = model.par.p1*x1 + model.par.p2*x2 + model.par.p3*x3
    print(f's1={s1_now:.2f}, w={w_now:.2f}: x1={x1:6.3f}, x2={x2:6.3f}, x3={x3:6.3f}, expenditure={expenditure:.6f}')


s1=0.00, w=0.00: x1= 0.000, x2= 0.000, x3= 6.667, expenditure=10.000000
s1=0.30, w=0.50: x1= 3.000, x2= 3.500, x3= 2.333, expenditure=10.000000
s1=0.60, w=0.70: x1= 6.000, x2= 2.800, x3= 0.800, expenditure=10.000000
s1=1.00, w=1.00: x1=10.000, x2= 0.000, x3= 0.000, expenditure=10.000000


In [7]:
# Thongs sanity check
model = ConsumerClass()

grid_sol = model.solve_grid(N=500)

print("Grid solution:")
print("s1 =", grid_sol.s1)
print("w  =", grid_sol.w)
print("s2 =", grid_sol.s2)
print("s3 =", grid_sol.s3)
print("u  =", grid_sol.u)

Grid solution:
s1 = 0.5350701402805611
w  = 0.438877755511022
s2 = 0.20404737330372166
s3 = 0.26088248641571726
u  = 3.40167436585882


Compare with L-BFGS-B

In [3]:
sol = model.solve()

print("\nL-BFGS-B solution:")
print("s1 =", sol.s1)
print("w  =", sol.w)
print("s2 =", sol.s2)
print("s3 =", sol.s3)
print("u  =", sol.u)


L-BFGS-B solution:
s1 = 0.5356233806953101
w  = 0.43947902810370126
s2 = 0.20408378532610758
s3 = 0.26029283397858227
u  = 3.401679875985606


In [4]:
sol.s1 + sol.s2 + sol.s3

np.float64(1.0)

### 1.2 Calibration


In [2]:
model_sub = ConsumerClass(par={'sigma_B': 3.0})

### 1.3 How do you know your answer is right?

There is no formula for the solution of this model, so you cannot look the answer up. Two cheap checks: 

1. Is the answer possible? All three shares should be strictly between 0 and 1 aand sum to one, and all three quantities should be positive

In [7]:
# Check that the solution is possible

print("Budget shares:")
print("s1 =", sol.s1)
print("s2 =", sol.s2)
print("s3 =", sol.s3)

print("\nSum of shares:")
print(sol.s1 + sol.s2 + sol.s3)

print("\nQuantities:")
x1, x2, x3 = model.quantities(sol.s1, sol.w)

print("x1 =", x1)
print("x2 =", x2)
print("x3 =", x3)

Budget shares:
s1 = 0.5356233806953101
s2 = 0.20408378532610758
s3 = 0.26029283397858227

Sum of shares:
1.0

Quantities:
x1 = 5.356233806953101
x2 = 2.040837853261076
x3 = 1.7352855598572152


In [8]:
0 < sol.s1 < 1
0 < sol.s2 < 1
0 < sol.s3 < 1

np.True_

In [9]:
sol.s1 + sol.s2 + sol.s3

np.float64(1.0)

In [10]:
x1 > 0
x2 > 0
x3 > 0

np.True_

In [11]:
print("All shares between 0 and 1:",
      0 < sol.s1 < 1 and
      0 < sol.s2 < 1 and
      0 < sol.s3 < 1)

print("Shares sum to 1:",
      np.isclose(sol.s1 + sol.s2 + sol.s3, 1))

print("All quantities positive:",
      x1 > 0 and x2 > 0 and x3 > 0)

All shares between 0 and 1: True
Shares sum to 1: True
All quantities positive: True


2. Do two different methods agree? In section 2 you solve the same problem twice.

In [12]:
grid_sol = model.solve_grid(N=1000)
sol = model.solve()

# Grid search:
print("Grid search:")
print("s1 =", grid_sol.s1)
print("w  =", grid_sol.w)
print("u  =", grid_sol.u)

# L-BFGS-B optimization:
print("\nL-BFGS-B:")
print("s1 =", sol.s1)
print("w  =", sol.w)
print("u  =", sol.u)

Grid search:
s1 = 0.5355355355355356
w  = 0.4394394394394394
u  = 3.4016797975402664

L-BFGS-B:
s1 = 0.5356233806953101
w  = 0.43947902810370126
u  = 3.401679875985606


The grid search only consider a finite number of points, so its result will not necessary be identical to L-BFGS-B. But with a sufficient fine grid, the results should be almost identical. 

In [13]:
# Calculate the difference 

print("Difference in s1:",
      abs(grid_sol.s1 - sol.s1))

print("Difference in w:",
      abs(grid_sol.w - sol.w))

print("Difference in utility:",
      abs(grid_sol.u - sol.u))

Difference in s1: 8.784515977455776e-05
Difference in w: 3.958866426184704e-05
Difference in utility: 7.844533955747579e-08


In [ ]:
grid_sol = model.solve_grid(N=1000)
sol = model.solve()

# Grid search:
print("Grid search:")
print("s1 =", grid_sol.s1)
print("w  =", grid_sol.w)
print("u  =", grid_sol.u)

# L-BFGS-B optimization:
print("\nL-BFGS-B:")
print("s1 =", sol.s1)
print("w  =", sol.w)
print("u  =", sol.u)

## 2. Solving the model numerically
We solve eq. 4 twice, in two very different ways


ModuleNotFoundError: No module named 'grid_solve'

### 2.1 A two-dimensional grid search
In this task we utilize our knowledge from the optimization lecture: We lay a grid over everything the
consumer can afford, compute utility in every point, and keep the best one. 


##### 2.1.1 
Make a grid of N values of s1 and N values of w, both from 0 to 1.
Compute utility in every combination, and keep the best point.
Implement it as solve_grid ().
Report both (s1, w) and the three budget shares it implies.
